# Chapter 10 lab — Which still-running worker may write?

Draft educator companion v1 · 8 September 2026 · 90 minutes.

Read [the chapter](https://www.profrod.ai/book/ch10-worker-recovery) alongside this lab. **Prerequisites:** Chapter 9 effects; generation numbers, leases and comparisons at the write boundary.

You will build one explicitly scoped decision function, challenge it with independently authored cases, and trace the same concern through the cumulative runtime. The manuscript is where you build the full components; this notebook is a focused companion, not a claim that importing a runtime teaches its construction.

**Before running:** write your prediction. Keep the worked solution below closed until you have attempted the function. Download/open this notebook in an existing Jupyter environment using the book’s Python 3.14 interpreter after completing repository setup in the book conventions. Unlike the two Chapter 1 notebooks, this lab requires the local checkout and its locked book dependencies. It does not install packages, launch a hosted notebook, or require model credentials.

**Without a notebook server:** read and edit the cells in your editor, then run `uv run --python 3.14 python book/always_on/educator/run_lesson_v1.py --chapter 10 --output /tmp/lucy-ch10-class.json` from the repository root. Use a new output filename on each retained run. The runner executes the saved notebook and records student results separately from the worked example.


## 1. Predict (10 minutes)

Worker A is still alive after its lease expires. Worker B claims generation two. Should A’s process liveness allow it to append a transcript or send an order?

Write both the expected result and the evidence that could disprove your explanation.


In [ ]:
import copy
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("Use the book Python 3.14 environment for Chapters 2–16.")
# Open the notebook inside your source checkout, or set this path explicitly.
start = Path(os.environ.get("SOVEREIGN_AGENT_REPO", Path.cwd())).resolve()
ROOT = next(
    (p for p in (start, *start.parents) if (p / "book/always_on/checkpoints/ch10.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Set SOVEREIGN_AGENT_REPO to the Sovereign Agent checkout.")
CHECKPOINT = ROOT / "book/always_on/checkpoints/ch10.py"
EXPECTED_CHECKPOINT_SHA256 = "f4fd29c6acba491b3348e71c9b95e9dc3c26d7f890dc9181c7bce74a840585ed"
if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError("Checkpoint version differs from this lesson; use its matching release.")
print("Chapter 10 checkpoint bytes match this lesson. No model or channel has been called.")

## 2. Build your decision (25 minutes)

Implement decide(case). Compare presented and current dictionaries on work, owner, generation and epoch. Return CURRENT only if all match, current.status is RUNNING, cancelled is false and now < expires; otherwise STALE. Inputs are trusted typed records. Equality with expiry is already expired.

`decide` is your code. `grade` and the fixtures are supplied test infrastructure. The examples below specify expected answers independently; do not generate those answers with your function. An unimplemented starter is reported as NOT_SUBMITTED, never as a pass.


In [ ]:
def grade(candidate, cases):
    results = []
    for index, (case, expected) in enumerate(cases, 1):
        supplied = copy.deepcopy(case)
        try:
            observed = candidate(supplied)
        except NotImplementedError:
            results.append({"case": index, "status": "NOT_SUBMITTED"})
            continue
        except Exception as error:
            results.append({"case": index, "status": "FAILED", "error_type": type(error).__name__})
            continue
        try:
            passed = json.dumps(observed, sort_keys=True, allow_nan=False) == json.dumps(
                expected, sort_keys=True, allow_nan=False
            ) and json.dumps(supplied, sort_keys=True, allow_nan=False) == json.dumps(
                case, sort_keys=True, allow_nan=False
            )
        except (TypeError, ValueError):
            passed = False
        results.append(
            {
                "case": index,
                "status": "PASS" if passed else "FAILED",
                "expected": expected,
                "observed": observed,
            }
        )
    return results


def decide(case):
    # Replace this body with your implementation of the contract above.
    raise NotImplementedError("Write your function before consulting the worked solution.")

In [ ]:
CASES = [
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "CURRENT",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "B",
                "generation": 2,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 100,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": True,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
]
submission_results = grade(decide, CASES)
print(json.dumps(submission_results, indent=2))

## 3. Inspect and run the cumulative reference (20 minutes)

checkpoints/ch10.py: old_worker and experiment. Follow readiness, actual expiry, replacement generation and the three refused old-worker actions.

Open the named code before running it. Point to where an input reaches a decision and where that decision changes an observable result. The next cell executes the supplied chapter checkpoint; it is reference evidence, not a substitute for your implementation. Local supplier and worker processes use temporary state and are cleaned up by the checkpoint. Chapter 11 also launches a bounded local MCP process. Chapter 15 does not install a system service.


In [ ]:
# This supplied cumulative program is separate from grading your function.
# It uses fixture models/channels. Some chapters start local child processes.
# No --live, --telegram or --containers switch is added.
reference_environment = {
    k: v for k, v in os.environ.items() if k in {"PATH", "SYSTEMROOT", "TMPDIR", "LANG", "LC_ALL"}
}
reference_environment["PYTHONPATH"] = str(ROOT / "src")
reference_run = subprocess.run(
    [sys.executable, str(CHECKPOINT)],
    cwd=ROOT,
    env=reference_environment,
    capture_output=True,
    text=True,
    timeout=180,
    check=True,
)
print(reference_run.stdout)
EXPECTED_OBSERVATIONS = ["Live stale worker refused: 3", "New model calls during recovery: 0"]
assert all(text in reference_run.stdout for text in EXPECTED_OBSERVATIONS)
print("REFERENCE_CHECKPOINT_PASSED — this is not your submission grade.")

## 4. Transfer the rule (20 minutes)

Create one case for a changed database epoch with unchanged owner and generation, and one exactly at lease expiry. Then find the real stale-worker checks at transcript, completion and supplier boundaries.

Add your new case to TRANSFER_CASES, with an independently calculated expected answer. A blank list means the transfer remains unsubmitted. Describe one limit of your function before comparing it with the runtime.


In [ ]:
TRANSFER_CASES = []  # Add (input, expected) pairs after writing your prediction.
transfer_results = grade(decide, TRANSFER_CASES)
print(json.dumps(transfer_results, indent=2) if transfer_results else "TRANSFER_NOT_SUBMITTED")

## 5. Worked solution — reveal after attempting the task

The following function is an answer key, not a replacement for your submission. Its case results are recorded separately. Your teacher grades your original function, explanation and transfer case.

Both cases are STALE. The checkpoint keeps one old worker alive and independently kills another; replacements use the existing approved operation with zero new model calls. Checking once at task start would leave a stale write window.


In [ ]:
def worked_decide(case):
    current, presented = case["current"], case["presented"]
    same = all(current[key] == presented[key] for key in ("work", "owner", "generation", "epoch"))
    valid = (
        same
        and current["status"] == "RUNNING"
        and not current["cancelled"]
        and case["now"] < current["expires"]
    )
    return "CURRENT" if valid else "STALE"


worked_results = grade(worked_decide, CASES)
assert all(row["status"] == "PASS" for row in worked_results)
print("WORKED_EXAMPLE_PASSED; submission_results remains separate.")

## 6. Break the tempting implementation (10 minutes)

Explain why the following shortcut violates at least one case. Predict which case catches it before running. Then name a different defect the current cases might miss.


In [ ]:
def tempting_shortcut(case):
    return "CURRENT"


shortcut_results = grade(tempting_shortcut, CASES)
assert any(row["status"] == "FAILED" for row in shortcut_results)
print(json.dumps(shortcut_results, indent=2))

## Exit ticket (5 minutes)

Submit your prediction, original decide function, case results, one transfer case, and the runtime path you traced. Explain: (1) which boundary Python enforced, (2) what evidence came from the supplied program, and (3) what remains unproved.

**Misconception to resolve:** Process liveness, current authority and external completion answer different questions. A lease is not a mechanism for undoing a call already sent.

**Scope of this lab:** The function models a predicate; real checking and the mediated write must share the correct transactional boundary. The experiment is local, not proof of arbitrary distributed fencing.

A successful reference run or worked example does not establish learner mastery. Instructor guidance and answers are in the matching versioned guide.
